This is the main analysis program designed by Warith

Make sure to run each code block in the order they appear otherwise the program will not work

Import the main libraries to be used

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler

Import kagglehub to download the dataset

In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shashwatwork/knee-osteoarthritis-dataset-with-severity", output_dir="data")

print("Path to dataset files:", path)

100%|██████████| 204M/204M [00:06<00:00, 30.6MB/s] 

Extracting files...


Path to dataset files: data


Setting up Matplotlib for visualisation

In [1]:
plt.rc('image', cmap='gray')
plt.rc('figure', autolayout=True)

Preprocessing

In [ ]:
def preprocess(img):
    # Define the different paths
    data_dir = Path("data")

    train_dir = data_dir / "train"
    val_dir = data_dir / "val"

    # Loads, resizes and batches images directly from paths

    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir,
        image_size=(224,224),
        batch_size=32,
        shuffle=True,
        color_mode="grayscale"
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir,
        image_size=(224,224),
        batch_size=32,
        shuffle=True,
        color_mode="grayscale"
    )

Define a convolution kernel
- This makes it possible to extract important image features
- In this case the vital parts of the X-Ray (The bones)

In [ ]:
kernel = tf.constant([
    [-1, -1, -1],
    [-1, 9, -1],
    [-1, -1, -1]
], dtype=tf.float32, shape=[3, 3, 1, 1])

Apply the convolution layer by grabbing a batch from the conveyor belt

In [ ]:
for images, labels in train_ds.take(1):
    # convert image tensor to float32
    images_float = tf.cast(images, tf.float32)

    # Apply sharpening kernel
    # strides=[1,1,1,1] moves pixel by pixel
    images_float = tf.nn.conv2d(images_float, kernel, strides=[1,1,1,1], padding="SAME")

    # plots original X-ray vs sharpened feature map side-by-side
    plt.figure(figsize=(10,10))
    # Original
    plt.subplot(1,2,1)
    plt.imshow(images[0], cmap="gray")
    plt.title("Original X-Ray")
    # Sharpened
    plt.subplot(1,2,2)
    plt.imshow(images_float[0], cmap="gray")
    plt.title("Sharpened X-Ray")

    plt.show()
    break